# AnimateDiff — text to short video with 🤗 Diffusers

AnimateDiff adds a *motion adapter* to an ordinary Stable Diffusion 1.5 checkpoint so it generates a short clip instead of a single frame. This follows the Diffusers AnimateDiff guide: motion adapter `guoyww/animatediff-motion-adapter-v1-5-2` on the SD1.5 fine-tune `SG161222/Realistic_Vision_V5.1_noVAE`. 16 frames at 512×512 fits a free T4 in fp16 with CPU offload.

Part of the SoapBox GenAI school (hathor.soapbox.community/colab). Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!pip -q install -U diffusers transformers accelerate

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
from diffusers.utils import export_to_gif

adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
model_id = "SG161222/Realistic_Vision_V5.1_noVAE"
pipe = AnimateDiffPipeline.from_pretrained(model_id, motion_adapter=adapter, torch_dtype=torch.float16)
pipe.scheduler = DDIMScheduler.from_pretrained(
    model_id, subfolder="scheduler",
    clip_sample=False, timestep_spacing="linspace", beta_schedule="linear", steps_offset=1,
)
pipe.enable_vae_slicing()
pipe.enable_model_cpu_offload()

In [ ]:
output = pipe(
    prompt="golden sunrise over the Nile, reeds moving in the wind, cinematic, high quality",
    negative_prompt="bad quality, worse quality, low resolution",
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=25,
    generator=torch.Generator("cpu").manual_seed(42),
)
export_to_gif(output.frames[0], "animation.gif")

In [ ]:
from IPython.display import Image
Image(filename="animation.gif")